[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00


In [2]:
import torch
import math

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    # pass
                  # pair i=0           pair i=1.  -> this across all x columns (D in total)
#                 (x_0, x_1)         (x_2, x_3)
#                 freq_0 = 1         freq_1 = 1/10000^(0.5)

# This pos across each x's S elements
# token pos=0:    angle = 0*1 = 0    angle = 0*freq_1 = 0
# token pos=1:    angle = 1*1 = 1    angle = 1*freq_1
# token pos=2:    angle = 2*1 = 2    angle = 2*freq_1
    B, S, D = q.shape
    idx = torch.arange(D//2).unsqueeze(0)  #0, 1, ... D/2   .repeat_interleave(2)
    # (1, D/2)
    pos = torch.arange(S).unsqueeze(1).float() # 0,1,2,... S
    # (S, 1)
    inv_freq = 1 / 10000**(2*idx / D)
    angle = pos @ inv_freq #-> (S, D/2)
    cos_ = torch.cos(angle)
    sin_ = torch.sin(angle)
    output = []
    for x in [q, k]:
      x0 = x[:, :, ::2]
      x1 = x[:, :, 1::2]
      x0 = x0*cos_ - x1*sin_
      x1 = x0*sin_ + x1*cos_
      output.append(x)
    return tuple(output)







In [11]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [12]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (3.9ms)
  ✅ [2/4] Preserves norm (7.8ms)
  ✅ [3/4] Relative position property (14.4ms)
  ✅ [4/4] Gradient flow (12.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (38.8ms total)
  Progress saved. Run status() to see your dashboard.

